# Active Directory Defensive Lab (Simulation)
**IBM SkillsBuild Cyber Security 2026 project**

This notebook demonstrates defensive review of fictional Windows security events. It does not connect to Active Directory, conduct an attack, or change Group Policy. The events and results are simulated for learning.

## 1. Imports and fictional sample data

In [ ]:
from collections import Counter
import json
from pathlib import Path

SAMPLE_EVENTS = [
    {"timestamp":"2026-10-20T09:14:00", "event_id":4769, "account":"svc_backup", "service":"MSSQLSvc/DB01.example.test", "ticket_encryption":"RC4", "source_host":"WS-014"},
    {"timestamp":"2026-10-20T09:15:00", "event_id":4769, "account":"svc_backup", "service":"MSSQLSvc/DB01.example.test", "ticket_encryption":"RC4", "source_host":"WS-014"},
    {"timestamp":"2026-10-20T09:16:00", "event_id":4769, "account":"svc_reports", "service":"HTTP/APP01.example.test", "ticket_encryption":"AES256", "source_host":"WS-021"},
    {"timestamp":"2026-10-20T09:22:00", "event_id":4624, "account":"analyst1", "service":"Interactive logon", "ticket_encryption":"N/A", "source_host":"WS-021"}
]
print(f"Loaded {len(SAMPLE_EVENTS)} fictional sample events.")

## 2. Detection rules for analyst review
These simple rules flag RC4-encrypted service-ticket events and service-account ticket requests for review. A flag is only a lead, not proof of malicious activity.

In [ ]:
def flag_event(event):
    flags = []
    if event.get('event_id') == 4769 and event.get('ticket_encryption') == 'RC4':
        flags.append('Review RC4-encrypted service ticket')
    if event.get('event_id') == 4769 and event.get('account', '').lower().startswith('svc_'):
        flags.append('Review service-account ticket request')
    return flags

def analyze_events(events):
    reviewed = []
    for event in events:
        item = dict(event)
        item['review_flags'] = flag_event(event)
        reviewed.append(item)
    return {
        'total_events': len(events),
        'event_id_counts': dict(Counter(str(e.get('event_id', 'unknown')) for e in events)),
        'events_with_review_flags': sum(bool(e['review_flags']) for e in reviewed),
        'reviewed_events': reviewed,
    }

results = analyze_events(SAMPLE_EVENTS)
print('Total sample events:', results['total_events'])
print('Events with review flags:', results['events_with_review_flags'])
print('Event ID counts:', results['event_id_counts'])
for event in results['reviewed_events']:
    if event['review_flags']:
        print('\n', event['timestamp'], '|', event['account'], '|', event['service'])
        for flag in event['review_flags']:
            print(' -', flag)

## 3. Defensive hardening checklist
Use these as review recommendations; no settings are changed by this notebook.

In [ ]:
checklist = [
    'Inventory service accounts and their SPNs',
    'Use long, unique, managed service-account credentials where supported',
    'Prefer AES encryption and assess legacy RC4 dependencies',
    'Apply least privilege to service accounts',
    'Review and test GPO changes before broad deployment',
    'Enable suitable auditing and monitor unusual service-ticket patterns',
]
for item in checklist:
    print('[ ]', item)

## 4. Export results
The JSON file contains only the fictional sample analysis.

In [ ]:
output_file = Path('simulated_analysis_report.json')
output_file.write_text(json.dumps(results, indent=2), encoding='utf-8')
print('Saved:', output_file.resolve())
print('\nImportant: all input data is fictional. This is a learning simulation, not evidence of a real attack.')